# Sheet 01: Understanding the Dataset

**Dataset:** EARS-Net antimicrobial resistance surveillance data (ECDC, via Kaggle).
Each row is one observation: for a given **country**, **year**, **bacterium**, and
**antibiotic**, it records **n_isolates** (number of bacterial samples tested) and
**pct_resistant** (% of those samples resistant to the antibiotic).

**Goal of this notebook:** inspect the raw structure, size, data types, and missing
values before cleaning.

## Key Terms (Glossary)

Understanding these domain terms is essential to interpreting the dataset:

- **Isolate** — a single bacterial sample taken from one patient's infection and
  grown in a lab so it can be tested against antibiotics.

- **n_isolates** — the *number of isolates tested* for a given
  country–year–bacterium–antibiotic combination. Higher numbers mean the result
  is based on more samples and is therefore more statistically reliable.

- **Resistant** — a bacterium is *resistant* to an antibiotic when it survives
  exposure to that drug. Resistant infections are dangerous because the standard
  medicine no longer works.

- **pct_resistant (%R)** — the *percentage of tested isolates that were resistant*
  to the antibiotic. Example: `pct_resistant = 9.8` means 9.8% of the samples
  survived the drug. Range in this dataset: 0–95%.

- **Antimicrobial Resistance (AMR)** — the broad phenomenon of microbes (bacteria
  here) evolving to survive the drugs designed to kill them. AMR is a major global
  public-health threat, which is what makes this dataset meaningful.

- **bacterium** — the type of bacteria being tested (e.g. *Escherichia coli*).

- **antibiotic** — the drug class the bacteria were tested against
  (e.g. *Fluoroquinolones*, *Carbapenems*).

- **Country / Year** — the reporting country (30 EU/EEA nations) and the year of
  the surveillance data (2010–2014).

In [14]:
import pandas as pd
import numpy as np

# Load the clean dataset we built in Stage 2
df = pd.read_csv("../data/raw/ears_net.csv")

df.head()

,country,year,bacterium,antibiotic,n_isolates,pct_resistant
0,Austria,2012,Acinetobacter spp.,Aminoglycosides,NaN,NaN
1,Austria,2013,Acinetobacter spp.,Aminoglycosides,51.0,9.80000
2,Austria,2014,Acinetobacter spp.,Aminoglycosides,79.0,8.86076
3,Belgium,2012,Acinetobacter spp.,Aminoglycosides,NaN,NaN
4,Belgium,2013,Acinetobacter spp.,Aminoglycosides,3.0,NaN


In [15]:
df.shape        # (rows, columns) - how big is the dataset?

(3628, 6)

**Size:** 3,628 rows × 6 columns. This is ample data for modelling —
each row is a country–year–bacterium–antibiotic resistance observation
covering 30 countries, 8 bacteria, 13 antibiotics, and years 2010–2014.

In [16]:
df.info()       # column types + non-null counts (reveals missing data)

<class 'pandas.DataFrame'>
RangeIndex: 3628 entries, 0 to 3627
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   country        3628 non-null   str    
 1   year           3628 non-null   int64  
 2   bacterium      3628 non-null   str    
 3   antibiotic     3628 non-null   str    
 4   n_isolates     3518 non-null   float64
 5   pct_resistant  3422 non-null   float64
dtypes: float64(2), int64(1), str(3)
memory usage: 170.2 KB


**Data types & completeness:** The four descriptor columns (country, year,
bacterium, antibiotic) are fully populated. The two numeric columns have gaps:
`n_isolates` has 3,518 non-null (110 missing) and `pct_resistant` has 3,422
non-null (206 missing). Types are correct — text for descriptors, numeric for
the measurements.

In [17]:
df.describe()   # statistics for the numeric columns

,year,n_isolates,pct_resistant
count,3628.000000,3518.000000,3422.000000
mean,2012.095921,917.297612,20.086091
std,1.399773,1478.133614,19.776429
min,2010.000000,0.000000,0.000000
25%,2011.000000,114.000000,5.000000
50%,2012.000000,374.500000,13.500000
75%,2013.000000,955.000000,29.200000
max,2014.000000,10349.000000,95.285360


**Statistics:** `pct_resistant` ranges from 0 to ~95% (mean ≈ 20%), which is
realistic for resistance data — no impossible values (e.g. no >100% or 999
data-entry errors). `n_isolates` is heavily right-skewed: median ≈ 374 but mean
≈ 917 and max = 10,349, meaning a few large countries test far more samples
than most.

In [18]:
df.isnull().sum()   # exactly how many missing values per column

country            0
year               0
bacterium          0
antibiotic         0
n_isolates       110
pct_resistant    206
dtype: int64

**Missing values:** 110 missing in `n_isolates`, 206 missing in `pct_resistant`;
descriptor columns have none. These are **not data errors** — ECDC does not
report a resistance percentage when fewer than ~10 isolates were tested, because
the sample is too small to be statistically reliable. Missingness therefore
flags low-sample combinations, and will be handled in the cleaning stage.

In [19]:
# How many rows for each bacterium? (target-balance style check)
df['bacterium'].value_counts()

bacterium
Escherichia coli            900
Pseudomonas aeruginosa      900
Klebsiella pneumoniae       750
Acinetobacter spp.          348
Streptococcus pneumoniae    290
Enterococcus faecium        150
Staphylococcus aureus       150
Enterococcus faecalis       140
Name: count, dtype: int64

**Bacteria distribution:** The bacteria are unevenly represented (E. coli: 900
rows down to E. faecalis: 140). This is expected — each bacterium is screened
against a different number of antibiotics, so this reflects the **feature**
distribution, not a target imbalance. The prediction target (`resistance_level`)
will be engineered in the next notebook.